In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
from sodapy import Socrata
import geopandas as gpd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

BASE_DIR      = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = os.path.join(BASE_DIR, "Scripts Python", "webpage_climate", "data")

os.chdir(BASE_DIR)
print("Directorio de trabajo :", os.getcwd())
print("Carpeta web/data      :", WEB_DATA_DIR)

Directorio de trabajo : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data
Carpeta web/data      : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data


In [2]:
pip install geopandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
pip install sodapy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## Datos diarios – Extracción API y cruce con alertas históricas

In [4]:
# Carga alertas históricas generadas por datos precipitacion historicos.ipynb
out_estaciones = os.path.join(WEB_DATA_DIR, "alerta_historica_estaciones.csv")
estaciones_alerta = pd.read_csv(out_estaciones)

print(f"Estaciones históricas cargadas: {len(estaciones_alerta):,}")
print("\n=== Distribución de alerta compuesta (histórica) ===")
estaciones_alerta.head()

Estaciones históricas cargadas: 1,860

=== Distribución de alerta compuesta (histórica) ===


,CodigoEstacion,NombreEstacion,Departamento,Municipio,Latitud,Longitud,frecuencia_extremos,pendiente,p_valor,tendencia,frecuencia_reciente,ratio_reciente,frecuencia_anio_corrido,frecuencia_ultimos_6m,ratio_anio_corrido,ratio_ultimos_6m,condicion_lluvia_6m,alerta_lluvias,sequia_categoria,cod_norm
0,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,5.412000,-76.418000,0.060748,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sin_datos,BAJA,NORMAL,11017020
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,5.888719,-76.145167,0.053353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sin_datos,BAJA,NORMAL,11025501
2,11027030,EL SIETE,CHOCO,EL CARMEN,5.862000,-76.152056,0.002809,-0.000307,0.180648,estable,0.001499,0.533733,0.0,0.0,0.0,0.0,DEFICIT,BAJA,LEVE,11027030
3,11027030,EL SIETE - AUT,CHOCO,EL CARMEN,5.862000,-76.152056,0.076628,-0.000307,0.180648,estable,0.001499,0.019565,0.0,0.0,0.0,0.0,DEFICIT,BAJA,LEVE,11027030
4,11027070,BORAUDO,CHOCÓ,LLORÓ,5.515000,-76.576000,0.051806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sin_datos,BAJA,NORMAL,11027070


In [5]:
# Carga davipola y proyecta a EPSG 3116 (necesario para sjoin con municipios)
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

print(f"Municipios cargados: {len(gdf_mun):,}")

Municipios cargados: 1,121


## Datos en tiempo real – API IDEAM

In [6]:
DATASET_ID = "s54a-sgyg"
client = Socrata("www.datos.gov.co", None)

# ORDER BY DESC es más confiable que max() para timestamps en Socrata
latest = client.get(DATASET_ID, select="fechaobservacion", order="fechaobservacion DESC", limit=1)
fecha_mapa = latest[0]["fechaobservacion"][:10]
print(f"Última fecha disponible: {fecha_mapa}")

where = (
    f"fechaobservacion >= '{fecha_mapa}T00:00:00' "
    f"AND fechaobservacion < '{fecha_mapa}T23:59:59.999'"
)

records, offset = [], 0
while True:
    batch = client.get(DATASET_ID, where=where, limit=100_000, offset=offset)
    if not batch:
        break
    records.extend(batch)
    offset += 100_000
    print(f"  {len(records):,} registros descargados...")
client.close()

Última fecha disponible: 2026-08-17


  73,273 registros descargados...


In [7]:
estaciones_alerta.columns

Index(['CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio',
       'Latitud', 'Longitud', 'frecuencia_extremos', 'pendiente', 'p_valor',
       'tendencia', 'frecuencia_reciente', 'ratio_reciente',
       'frecuencia_anio_corrido', 'frecuencia_ultimos_6m',
       'ratio_anio_corrido', 'ratio_ultimos_6m', 'condicion_lluvia_6m',
       'alerta_lluvias', 'sequia_categoria', 'cod_norm'],
      dtype='object')

In [8]:
# Agrega lecturas a nivel de estación
df_api = pd.DataFrame.from_records(records)

for col in ("valorobservado", "latitud", "longitud"):
    df_api[col] = pd.to_numeric(df_api[col], errors="coerce")

df_api = df_api.dropna(subset=["latitud", "longitud", "valorobservado"])
df_api = df_api[df_api["valorobservado"] >= 0]

STATION_COLS_API = [
    "codigoestacion", "nombreestacion", "departamento",
    "municipio", "zonahidrografica", "latitud", "longitud",
]

df_dia_hoy = (
    df_api.groupby(STATION_COLS_API, as_index=False)
    .agg(
        precip_acum_mm=("valorobservado", "sum"),
        precip_max_10min=("valorobservado", "max"),
        n_lecturas=("valorobservado", "count"),
    )
)
del df_api

# Normaliza cod_norm en ambos lados
df_dia_hoy["cod_norm"] = df_dia_hoy["codigoestacion"].astype(str).str.strip().str.lstrip("0")
estaciones_alerta["cod_norm"] = estaciones_alerta["cod_norm"].astype(str).str.strip().str.lstrip("0")

# ── Base = TODAS las estaciones históricas; left join con datos del día ───────
cols_precip = ['cod_norm', 'precip_acum_mm', 'precip_max_10min', 'n_lecturas', 'zonahidrografica']

df_dia = (
    estaciones_alerta
    .rename(columns={
        'CodigoEstacion': 'codigoestacion',
        'NombreEstacion': 'nombreestacion',
        'Latitud':        'latitud',
        'Longitud':       'longitud',
        'Departamento':   'departamento',
        'Municipio':      'municipio',
    })
    .merge(df_dia_hoy[cols_precip], on='cod_norm', how='left')
)
del df_dia_hoy

# Rellenar faltantes (estaciones sin lectura hoy conservan sus indicadores históricos)
df_dia["alerta_lluvias"]      = df_dia["alerta_lluvias"].fillna("BAJA")
df_dia["frecuencia_extremos"] = df_dia["frecuencia_extremos"].fillna(0)
df_dia["frecuencia_reciente"] = df_dia["frecuencia_reciente"].fillna(0)
df_dia["ratio_reciente"]      = df_dia["ratio_reciente"].fillna(np.nan)
df_dia["tendencia"]           = df_dia["tendencia"].fillna("sin_datos")
df_dia["sequia_categoria"]    = df_dia["sequia_categoria"].fillna("NORMAL")
df_dia["precip_acum_mm"]      = df_dia["precip_acum_mm"].fillna(np.nan)
df_dia["precip_max_10min"]    = df_dia["precip_max_10min"].fillna(np.nan)
df_dia["n_lecturas"]          = df_dia["n_lecturas"].fillna(0).astype(int)

print(f"Estaciones históricas (base)   : {len(df_dia):,}")
print(f"  Con lectura hoy               : {(df_dia['n_lecturas'] > 0).sum():,}")
print(f"  Sin lectura hoy               : {(df_dia['n_lecturas'] == 0).sum():,}")

Estaciones históricas (base)   : 1,860
  Con lectura hoy               : 892
  Sin lectura hoy               : 968


In [9]:
df_dia.head()

,codigoestacion,nombreestacion,departamento,municipio,latitud,longitud,frecuencia_extremos,pendiente,p_valor,tendencia,frecuencia_reciente,ratio_reciente,frecuencia_anio_corrido,frecuencia_ultimos_6m,ratio_anio_corrido,ratio_ultimos_6m,condicion_lluvia_6m,alerta_lluvias,sequia_categoria,cod_norm,precip_acum_mm,precip_max_10min,n_lecturas,zonahidrografica
0,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,5.412000,-76.418000,0.060748,NaN,NaN,sin_datos,0.000000,NaN,NaN,NaN,NaN,NaN,sin_datos,BAJA,NORMAL,11017020,NaN,NaN,0,NaN
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,5.888719,-76.145167,0.053353,NaN,NaN,sin_datos,0.000000,NaN,NaN,NaN,NaN,NaN,sin_datos,BAJA,NORMAL,11025501,NaN,NaN,0,NaN
2,11027030,EL SIETE,CHOCO,EL CARMEN,5.862000,-76.152056,0.002809,-0.000307,0.180648,estable,0.001499,0.533733,0.0,0.0,0.0,0.0,DEFICIT,BAJA,LEVE,11027030,6.7,0.8,144,Atrato - Darién
3,11027030,EL SIETE - AUT,CHOCO,EL CARMEN,5.862000,-76.152056,0.076628,-0.000307,0.180648,estable,0.001499,0.019565,0.0,0.0,0.0,0.0,DEFICIT,BAJA,LEVE,11027030,6.7,0.8,144,Atrato - Darién
4,11027070,BORAUDO,CHOCÓ,LLORÓ,5.515000,-76.576000,0.051806,NaN,NaN,sin_datos,0.000000,NaN,NaN,NaN,NaN,NaN,sin_datos,BAJA,NORMAL,11027070,NaN,NaN,0,NaN


In [10]:
# ── Helpers ───────────────────────────────────────────────────────────────
NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}
INV_NIVEL      = {v: k for k, v in NIVEL_NUMERICO.items()}

def alerta_max(serie):
    """Nivel de alerta más alto entre las estaciones del municipio."""
    nums = serie.map(NIVEL_NUMERICO).dropna()
    return INV_NIVEL.get(int(nums.max()), 'BAJA') if not nums.empty else 'BAJA'

def modo_seguro(serie, default='BAJA'):
    vc = serie.dropna().value_counts()
    return vc.idxmax() if not vc.empty else default

def tendencia_muni(serie):
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'creciente' in vals.values:
        return 'creciente'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'sin_datos'

def modo_sequia(serie):
    return modo_seguro(serie, default='NORMAL')

# ── Sjoin invertido: cada ESTACIÓN → municipio más cercano ────────────────
# (antes era gdf_mun→gdf_dia, que daba sólo 1 estación por municipio)
gdf_dia = gpd.GeoDataFrame(
    df_dia.dropna(subset=['latitud', 'longitud']),
    geometry=gpd.points_from_xy(
        df_dia.dropna(subset=['latitud', 'longitud']).longitud,
        df_dia.dropna(subset=['latitud', 'longitud']).latitud,
    ),
    crs="EPSG:4326",
).to_crs(epsg=3116)

gdf_mun_cols = gdf_mun[['COD_MPIO', 'NOM_MPIO', 'NOM_DPTO', 'LATITUD', 'LONGITUD', 'geometry']].copy()

# Cada estación → municipio más cercano; varias estaciones pueden caer en el mismo municipio
df_muni = gpd.sjoin_nearest(gdf_dia, gdf_mun_cols, how="left", distance_col="dist_m")

# ── Agrupación a nivel municipal — alerta = peor nivel en el municipio ────
df_muni = (
    df_muni
    .groupby(["COD_MPIO", "NOM_MPIO", "NOM_DPTO", "LATITUD", "LONGITUD"], as_index=False)
    .agg(
        precip_acum_mm=("precip_acum_mm", "mean"),
        precip_max_10min=("precip_max_10min", "max"),
        n_estaciones=("codigoestacion", "count"),
        alerta_compuesta=("alerta_lluvias", alerta_max),       # MAX, no moda
        frecuencia_extremos=("frecuencia_extremos", "max"),
        frecuencia_reciente=("frecuencia_reciente", "max"),
        ratio_reciente=("ratio_reciente", "max"),
        tendencia=("tendencia", tendencia_muni),
        sequia_categoria=("sequia_categoria", modo_sequia),
    )
)

df_muni["alerta"] = df_muni["alerta_compuesta"].map(NIVEL_NUMERICO).fillna(0).astype(int)
df_muni["fecha"]  = fecha_mapa

# ── Agrega flag de afectación histórica por inundaciones ──────────────────
flood = pd.read_excel(
    os.path.join(BASE_DIR, "Datos Procesados", "municipios_afectados_ola_invernal.xlsx"),
    usecols=['cod_divipola', 'municipio_afectado']
)
flood['cod_divipola'] = flood['cod_divipola'].astype(str)
df_muni['COD_MPIO'] = df_muni['COD_MPIO'].astype(str)
df_muni = df_muni.merge(flood, left_on='COD_MPIO', right_on='cod_divipola', how='left').drop(columns='cod_divipola')
df_muni['municipio_afectado'] = df_muni['municipio_afectado'].fillna(0).astype(int)

print(f"Municipios con estaciones asignadas: {len(df_muni):,}")
print(f"\n=== Alerta compuesta por municipio (peor nivel) ===")
print(df_muni['alerta_compuesta'].value_counts())
print(f"\nTotal con alerta > BAJA: {(df_muni['alerta'] > 0).sum():,}")
print(f"Municipios con antecedente inundación: {(df_muni['municipio_afectado'] == 1).sum():,}")

out_path = os.path.join(WEB_DATA_DIR, "datos_municipios.csv")
df_muni.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\nGuardado:", out_path)
df_muni.sort_values('alerta', ascending=False).head(5)

Municipios con estaciones asignadas: 589

=== Alerta compuesta por municipio (peor nivel) ===
alerta_compuesta
MODERADA    231
BAJA        194
ALTA         93
CRÍTICA      71
Name: count, dtype: int64

Total con alerta > BAJA: 395
Municipios con antecedente inundación: 418

Guardado: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data\datos_municipios.csv


,COD_MPIO,NOM_MPIO,NOM_DPTO,LATITUD,LONGITUD,precip_acum_mm,precip_max_10min,n_estaciones,alerta_compuesta,frecuencia_extremos,frecuencia_reciente,ratio_reciente,tendencia,sequia_categoria,alerta,fecha,municipio_afectado
586,97001,MITÚ,VAUPÉS,1.061482,-70.466884,0.200,0.20,3,CRÍTICA,0.111913,0.140073,5.219003,estable,NORMAL,3,2026-08-17,0
0,5001,MEDELLÍN,ANTIOQUIA,6.257590,-75.611031,43.653,3.97,5,CRÍTICA,0.143770,0.147156,6.429899,estable,NORMAL,3,2026-08-17,1
559,81736,SARAVENA,ARAUCA,6.906942,-71.850708,NaN,NaN,3,CRÍTICA,0.106667,0.135940,5.256338,estable,LEVE,3,2026-08-17,1
531,76001,CALI,VALLE DEL CAUCA,3.399044,-76.576493,115.000,5.00,10,CRÍTICA,0.151429,0.175244,6.039795,creciente,NORMAL,3,2026-08-17,1
529,73870,VILLAHERMOSA,TOLIMA,4.965753,-75.155930,NaN,NaN,2,CRÍTICA,0.138298,0.154458,3.131681,estable,MODERADA,3,2026-08-17,1


## GeoJSON de municipios con polígonos para el dashboard

In [11]:

# ── Genera GeoJSON de municipios con polígonos para el dashboard ─────────────
import requests, zipfile, io

# ── 1. Geometría (cache local; solo se descarga una vez) ─────────────────────
geo_cache = os.path.join(WEB_DATA_DIR, 'municipios_colombia_geo.gpkg')

if os.path.exists(geo_cache):
    print("Cargando geometría desde caché local...")
    gdf_geo = gpd.read_file(geo_cache)
else:
    print("Descargando polígonos GADM Colombia nivel 2 (primera vez)...")
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_COL_2.json.zip"
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        fname = next(f for f in z.namelist() if f.endswith('.json'))
        with z.open(fname) as f:
            gdf_geo = gpd.read_file(f)
    gdf_geo = gdf_geo.to_crs('EPSG:4326')
    gdf_geo['geometry'] = gdf_geo['geometry'].simplify(tolerance=0.005, preserve_topology=True)
    gdf_geo = gdf_geo[['NAME_1', 'NAME_2', 'geometry']].copy()
    gdf_geo.to_file(geo_cache, driver='GPKG')
    print(f"  → {len(gdf_geo):,} polígonos guardados en caché ({geo_cache})")

# ── Asignar COD_MPIO (código DANE) a polígonos GADM si aún no está ───────────
if 'COD_MPIO' not in gdf_geo.columns:
    print("Asignando códigos DANE a polígonos GADM via sjoin...")
    gdf_mun_4326 = gdf_mun.to_crs('EPSG:4326')[['COD_MPIO', 'geometry']].copy()

    # Cada punto DANE cae dentro de su polígono GADM
    joined = gpd.sjoin(gdf_mun_4326, gdf_geo[['geometry']], how='left', predicate='within')
    code_map = (
        joined.dropna(subset=['index_right'])
        .groupby('index_right')['COD_MPIO']
        .first()
    )
    gdf_geo['COD_MPIO'] = gdf_geo.index.map(code_map)

    # Polígonos sin coincidencia → asignar por el punto más cercano
    missing = gdf_geo['COD_MPIO'].isna()
    if missing.any():
        near = gpd.sjoin_nearest(
            gdf_geo[missing][['geometry']].reset_index(),
            gdf_mun_4326.reset_index(drop=True),
            how='left',
        ).drop_duplicates('index').set_index('index')['COD_MPIO']
        gdf_geo.loc[missing, 'COD_MPIO'] = gdf_geo[missing].index.map(near)

    gdf_geo['COD_MPIO'] = pd.to_numeric(gdf_geo['COD_MPIO'], errors='coerce').apply(lambda x: str(int(x)) if pd.notna(x) else '').str.strip()
    gdf_geo.to_file(geo_cache, driver='GPKG')
    print(f"  → COD_MPIO asignado ({gdf_geo['COD_MPIO'].notna().sum():,} polígonos) y caché actualizado")

print(f"Polígonos disponibles: {len(gdf_geo):,}")

# ── 2. Preparar datos de alerta (df_muni está en memoria del paso anterior) ──
df_alert = df_muni.copy()
df_alert['COD_MPIO'] = df_alert['COD_MPIO'].astype(str).str.strip()

cols_join = [
    'COD_MPIO', 'NOM_MPIO', 'NOM_DPTO',
    'alerta', 'alerta_compuesta', 'precip_acum_mm', 'precip_max_10min',
    'n_estaciones', 'sequia_categoria', 'fecha', 'municipio_afectado',
    'frecuencia_extremos', 'frecuencia_reciente', 'tendencia',
]
df_alert = df_alert[[c for c in cols_join if c in df_alert.columns]]

# ── 3. Join por código DANE ──────────────────────────────────────────────────
gdf_geo['COD_MPIO'] = pd.to_numeric(gdf_geo['COD_MPIO'], errors='coerce').apply(lambda x: str(int(x)) if pd.notna(x) else '').str.strip()
gdf_out = gdf_geo.merge(df_alert, on='COD_MPIO', how='left')
matched = gdf_out['alerta'].notna().sum()
print(f"Polígonos con datos de alerta: {matched:,} / {len(gdf_out):,}")

# ── 4. Rellenar faltantes ─────────────────────────────────────────────────────
for col in ['precip_acum_mm', 'precip_max_10min', 'frecuencia_extremos', 'frecuencia_reciente', 'n_estaciones']:
    gdf_out[col] = pd.to_numeric(gdf_out[col], errors='coerce').fillna(0)
gdf_out['alerta']             = pd.to_numeric(gdf_out['alerta'], errors='coerce').fillna(-1).astype(int)
gdf_out['alerta_compuesta']   = gdf_out['alerta_compuesta'].fillna('SIN_DATOS')
gdf_out['sequia_categoria']   = gdf_out['sequia_categoria'].fillna('NORMAL')
gdf_out['tendencia']          = gdf_out['tendencia'].fillna('sin_datos')
gdf_out['municipio_afectado'] = pd.to_numeric(gdf_out['municipio_afectado'], errors='coerce').fillna(0).astype(int)
gdf_out['NOM_MPIO']           = gdf_out['NOM_MPIO'].fillna(gdf_out['NAME_2'])
gdf_out['NOM_DPTO']           = gdf_out['NOM_DPTO'].fillna(gdf_out['NAME_1'])

# ── 5. Exportar GeoJSON ───────────────────────────────────────────────────────
keep = ['NAME_1', 'NAME_2', 'COD_MPIO', 'NOM_MPIO', 'NOM_DPTO',
        'alerta', 'alerta_compuesta', 'precip_acum_mm', 'precip_max_10min',
        'n_estaciones', 'sequia_categoria', 'fecha', 'municipio_afectado',
        'frecuencia_extremos', 'frecuencia_reciente', 'tendencia', 'geometry']
gdf_final = gdf_out[[c for c in keep if c in gdf_out.columns]]

out_geo = os.path.join(WEB_DATA_DIR, 'municipios_alertas.geojson')
gdf_final.to_file(out_geo, driver='GeoJSON')
size_kb = os.path.getsize(out_geo) / 1024

print(f"\nGeoJSON exportado: {out_geo}")
print(f"  Tamaño        : {size_kb:.0f} KB")
print(f"  ALTA o CRÍTICA: {(gdf_final['alerta'] >= 2).sum():,} municipios")
print(f"  Sequía severa : {(gdf_final['sequia_categoria'].isin(['SEVERA','EXTREMA'])).sum():,} municipios")
gdf_final.sort_values('alerta', ascending=False).head(5)[['NOM_MPIO','NOM_DPTO','alerta_compuesta','precip_acum_mm']]


Cargando geometría desde caché local...
Polígonos disponibles: 1,119
Polígonos con datos de alerta: 591 / 1,119



GeoJSON exportado: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data\municipios_alertas.geojson
  Tamaño        : 1706 KB
  ALTA o CRÍTICA: 163 municipios
  Sequía severa : 18 municipios


,NOM_MPIO,NOM_DPTO,alerta_compuesta,precip_acum_mm
1110,MITÚ,VAUPÉS,CRÍTICA,0.200000
380,HATO COROZAL,CASANARE,CRÍTICA,13.800000
18,ANGOSTURA,ANTIOQUIA,CRÍTICA,0.000000
493,UNIÓN PANAMERICANA,CHOCÓ,CRÍTICA,12.097556
497,CERETÉ,CÓRDOBA,CRÍTICA,13.600000
